In [27]:
import pandas as pd
import numpy as np
import math

In [28]:
def entropy(labels):
    zeros = 0
    ones = 0

    for val in labels:
        if val == 0:
            zeros += 1
        else:
            ones += 1

    total = zeros + ones

    if total == 0:
        return 0

    p0 = zeros / total
    p1 = ones / total

    ent = 0

    if p0 > 0:
        ent -= p0 * math.log2(p0)
    if p1 > 0:
        ent -= p1 * math.log2(p1)

    return ent

In [29]:
def split_data(X, y, feature_index, threshold):
    left_X, right_X = [], []
    left_y, right_y = [], []

    for i in range(len(X)):
        if X[i][feature_index] <= threshold:
            left_X.append(X[i])
            left_y.append(y[i])
        else:
            right_X.append(X[i])
            right_y.append(y[i])

    return left_X, right_X, left_y, right_y

In [30]:
def information_gain(parent_y, left_y, right_y):
    total = len(parent_y)

    parent_entropy = entropy(parent_y)
    left_entropy = entropy(left_y)
    right_entropy = entropy(right_y)

    weight_left = len(left_y) / total
    weight_right = len(right_y) / total

    weighted_entropy = weight_left * left_entropy + weight_right * right_entropy

    return parent_entropy - weighted_entropy

In [31]:
def find_best_threshold(X, y, feature_index):
    
    values = sorted(list(set(X[:, feature_index])))

    best_threshold = None
    best_ig = -1

    for i in range(len(values) - 1):
        threshold = (values[i] + values[i+1]) / 2

        left_X, right_X, left_y, right_y = split_data(X, y, feature_index, threshold)

        ig = information_gain(y, left_y, right_y)

        print("Threshold:", threshold, "IG:", ig)

        if ig > best_ig:
            best_ig = ig
            best_threshold = threshold

    return best_threshold, best_ig

In [32]:
def majority_class(labels):
    zeros = 0
    ones = 0

    for val in labels:
        if val == 0:
            zeros += 1
        else:
            ones += 1

    return 0 if zeros > ones else 1

In [33]:
def build_tree(X, y):

    threshold_age, age_ig = find_best_threshold(X, y, 0)

    print("Best Age Threshold:", threshold_age)
    print("Best Age IG:", age_ig)

    # ADD THIS LINE - Actually perform the split!
    left_X, right_X, left_y, right_y = split_data(X, y, 0, threshold_age)

    ig_root = information_gain(y, left_y, right_y)
    print("Information Gain (Age):", ig_root)

    # SECOND SPLIT (Salary on left side)
    threshold_salary = 50000
    ll_X, lr_X, ll_y, lr_y = split_data(left_X, left_y, 1, threshold_salary)

    ig_left = information_gain(left_y, ll_y, lr_y)
    print("Information Gain (Salary | Age<=40):", ig_left)

    # Tree structure
    tree = {
        "feature": "Age",
        "threshold": threshold_age,
        "left": {
            "feature": "Salary",
            "threshold": threshold_salary,
            "left": majority_class(ll_y),
            "right": majority_class(lr_y)
        },
        "right": majority_class(right_y)
    }

    return tree

In [34]:
def predict(tree, sample):

    if sample[0] <= tree["threshold"]:
        if sample[1] <= tree["left"]["threshold"]:
            return tree["left"]["left"]
        else:
            return tree["left"]["right"]
    else:
        return tree["right"]

In [35]:
data = pd.read_csv('Social_Network_Ads.csv')

data['Gender'] = data['Gender'].replace({'Male': 0, 'Female': 1})

X = data[['Age', 'EstimatedSalary']].values
y = data['Purchased'].values

In [36]:
tree = build_tree(X, y)

print("\nDecision Tree:")
print(tree)

Threshold: 18.5 IG: 0.008041286196445951
Threshold: 19.5 IG: 0.019517697635218823
Threshold: 20.5 IG: 0.03126003999849525
Threshold: 21.5 IG: 0.03809425561823365
Threshold: 22.5 IG: 0.04676889127268935
Threshold: 23.5 IG: 0.05737875909332901
Threshold: 24.5 IG: 0.07372336899630028
Threshold: 25.5 IG: 0.08492143453377388
Threshold: 26.5 IG: 0.11605994078726267
Threshold: 27.5 IG: 0.11102523763898497
Threshold: 28.5 IG: 0.12339268614944143
Threshold: 29.5 IG: 0.13292741508942163
Threshold: 30.5 IG: 0.13530653492120637
Threshold: 31.5 IG: 0.14915417475165949
Threshold: 32.5 IG: 0.1237280723942702
Threshold: 33.5 IG: 0.13451103697839373
Threshold: 34.5 IG: 0.13932353441905976
Threshold: 35.5 IG: 0.19063162168485193
Threshold: 36.5 IG: 0.1787769775451883
Threshold: 37.5 IG: 0.17529187093060128
Threshold: 38.5 IG: 0.20524913593686578
Threshold: 39.5 IG: 0.20150202207429646
Threshold: 40.5 IG: 0.22791473320871125
Threshold: 41.5 IG: 0.28750965066448686
Threshold: 42.5 IG: 0.3063051536835881
T

In [37]:
sample = [30, 40000]

result = predict(tree, sample)

print("Sample:", sample)
print("Prediction:", result)

Sample: [30, 40000]
Prediction: 0
